<a href="https://colab.research.google.com/github/Somendar/dataviz-excercises-SomendarKumarDas/blob/main/lecture06_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [5]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/sample_data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         60  7168.0   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [6]:
# Task 1

# Filter fossil fuels only
fossil_df = df[df['Source_Type'] == 'Fossil'].copy()

# Treemap
fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map={
        'Coal': '#4E79A7',
        'Oil': '#F28E2B',
        'Natural Gas': '#59A14F'
    },
    title='Coal dominates fossil fuel generation in several countries'
)

# Show TWh values only
fig.update_traces(
    textinfo='label+value',
    root_color='lightgrey'
)

# Grey out parent nodes
fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25)
)

fig.show()


Coal remains the dominant fossil fuel electricity source across many countries and regions, contributing a much larger share of TWh generation than oil or natural gas in several markets. The treemap also highlights how fossil fuel production is concentrated among a relatively small number of countries.

## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [7]:
# Task 2

# Load tips dataset
tips = px.data.tips()

# Aggregate total bill
tips_grouped = (
    tips.groupby(['day', 'time', 'smoker'], as_index=False)['total_bill']
    .sum()
)

# Sunburst chart
fig = px.sunburst(
    tips_grouped,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map={
        'Yes': '#4E79A7',
        'No': '#F28E2B'
    },
    title='Dinner service contributes most total bill revenue across the week'
)

# Show dollar totals
fig.update_traces(
    textinfo='label+value',
    root_color='lightgrey'
)

fig.update_layout(
    margin=dict(t=50, l=25, r=25, b=25)
)

fig.show()

Dinner service generates the highest total bill revenue throughout the week, especially among non-smokers. Weekend evenings contribute a large share of restaurant sales, making dinner periods the strongest revenue driver overall.

## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [8]:
# Task 3 — charts

# Filter low-carbon sources
low_carbon = (
    df[df['Source_Type'] == 'Low-carbon']
    .groupby('Country', as_index=False)['TWh']
    .sum()
)

# Add dummy root node
low_carbon['All'] = 'Low-carbon'

# Treemap
fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='A few countries account for most low-carbon electricity generation'
)

fig_tree.update_traces(
    textinfo='label+value',
    root_color='lightgrey'
)

fig_tree.show()

# Horizontal bar chart
low_carbon_sorted = low_carbon.sort_values('TWh', ascending=True)

fig_bar = px.bar(
    low_carbon_sorted,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon electricity generation is easier to compare in a bar chart'
)

fig_bar.update_layout(
    xaxis_title='Low-carbon TWh',
    yaxis_title='Country',
    showlegend=False
)

fig_bar.show()


A small group of countries produces the majority of low-carbon electricity generation. While the treemap effectively shows contribution proportions, the horizontal bar chart makes cross-country comparisons and rankings much easier to interpret because bar lengths are more precise than area sizes.